In [ ]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report
from imblearn.over_sampling import SMOTE
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.initializers import GlorotUniform

# Paths
DATASET_DIR = "datasets/"
EXTERNAL_TEST_FILE = "TestFinal\ForCGP\external_test\TestBlackDataset.csv"  
dataset_files = [f for f in os.listdir(DATASET_DIR) if f.endswith(".csv")]

# Load external test data once
external_df = pd.read_csv(EXTERNAL_TEST_FILE)
external_df = external_df.drop(columns=['sample'], errors='ignore').dropna()
X_external = external_df.drop(columns=['stage']).values
y_external = external_df['stage'].values

# Model configurations
model_configs = {
    "simple": [128, 64],
    "complex": [256, 128, 64]
}

# Results storage
summary_results = []

for file in dataset_files:
    print(f"Processing: {file}")
    df = pd.read_csv(os.path.join(DATASET_DIR, file))
    df = df.drop(columns=['sample'], errors='ignore').dropna()

    X = df.drop(columns=['stage']).values
    y = df['stage'].values

    # Label encoding
    le = LabelEncoder()
    y_encoded = le.fit_transform(y)
    y_categorical = to_categorical(y_encoded)
    labels = le.classes_

    # Scaling
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # Split: 80% train, 10% val, 10% test
    X_temp, X_test, y_temp, y_test = train_test_split(
        X_scaled, y_categorical, test_size=0.1, random_state=42, stratify=y_categorical
    )
    X_train, X_val, y_train, y_val = train_test_split(
        X_temp, y_temp, test_size=1/9, random_state=42, stratify=y_temp
    )

    # SMOTE on training set
    sm = SMOTE(random_state=42)
    y_train_labels = np.argmax(y_train, axis=1)
    X_train_sm, y_train_sm = sm.fit_resample(X_train, y_train_labels)
    y_train_sm = to_categorical(y_train_sm)

    input_dim = X_train.shape[1]
    output_dim = y_categorical.shape[1]

    for config_name, layers in model_configs.items():
        print(f"  Training model: {config_name}")
        model = Sequential()
        initializer = GlorotUniform()

        for i, units in enumerate(layers):
            if i == 0:
                model.add(Dense(units, input_dim=input_dim, activation='relu', kernel_initializer=initializer))
            else:
                model.add(Dense(units, activation='relu', kernel_initializer=initializer))
            model.add(BatchNormalization())
            model.add(Dropout(0.3))

        model.add(Dense(output_dim, activation='sigmoid'))
        model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

        early_stop = EarlyStopping(monitor='val_accuracy', patience=10, restore_best_weights=True)
        reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, verbose=0)

        model.fit(
            X_train_sm, y_train_sm,
            validation_data=(X_val, y_val),
            epochs=100,
            batch_size=32,
            verbose=0,
            callbacks=[early_stop, reduce_lr]
        )

        metrics_summary = {
            "Dataset": file,
            "Model_Config": config_name
        }

        # Train/Val/Test evaluation
        for split_name, X_split, y_split in [("Train", X_train, y_train), ("Val", X_val, y_val), ("Test", X_test, y_test)]:
            y_true = np.argmax(y_split, axis=1)
            y_pred_probs = model.predict(X_split, verbose=0)
            y_pred = np.argmax(y_pred_probs, axis=1)
            report = classification_report(y_true, y_pred, target_names=labels, output_dict=True, zero_division=0)

            metrics_summary[f"{split_name}_Precision"] = report["macro avg"]["precision"]
            metrics_summary[f"{split_name}_Recall"] = report["macro avg"]["recall"]
            metrics_summary[f"{split_name}_F1_Score"] = report["macro avg"]["f1-score"]
            metrics_summary[f"{split_name}_Accuracy"] = report["accuracy"]

        # External test evaluation (manually loaded earlier)
        try:
            y_ext_encoded = le.transform(y_external)
            y_ext_cat = to_categorical(y_ext_encoded)
            X_ext_scaled = scaler.transform(X_external)

            y_ext_pred_probs = model.predict(X_ext_scaled, verbose=0)
            y_ext_pred = np.argmax(y_ext_pred_probs, axis=1)
            y_ext_true = np.argmax(y_ext_cat, axis=1)

            report_ext = classification_report(
                y_ext_true, y_ext_pred, target_names=labels, output_dict=True, zero_division=0
            )

            metrics_summary["External_Precision"] = report_ext["macro avg"]["precision"]
            metrics_summary["External_Recall"] = report_ext["macro avg"]["recall"]
            metrics_summary["External_F1_Score"] = report_ext["macro avg"]["f1-score"]
            metrics_summary["External_Accuracy"] = report_ext["accuracy"]
        except Exception as e:
            print(f"External test evaluation failed: {e}")
            metrics_summary["External_Precision"] = np.nan
            metrics_summary["External_Recall"] = np.nan
            metrics_summary["External_F1_Score"] = np.nan
            metrics_summary["External_Accuracy"] = np.nan

        summary_results.append(metrics_summary)

# Save to CSV
summary_df = pd.DataFrame(summary_results)
summary_df.to_csv("BlackResult.csv", index=False)
print("All results saved to BlackResult.csv")